In [16]:
from pathlib import Path
from PIL import Image
import subprocess
import random
import shutil
import json
import re

FPS = 12
OUT_DIR = Path("frames")
OUT_DIR.mkdir(exist_ok=True)

assets = {
    "both_quiet": Image.open("left_and_right_silent.png").convert("RGB"),
    "left_open": Image.open("left_speaking.png").convert("RGB"),
    "right_open": Image.open("right_speaking.png").convert("RGB"),
    "left_blink": Image.open("left_blinks.png").convert("RGB"),
    "right_blink": Image.open("right_blinks.png").convert("RGB"),
}

# Load and parse exported timings from timings.js
raw_timings_js = Path("timings.js").read_text(encoding="utf-8")
match = re.search(r"export\s+const\s+timings\s*=\s*(\[[\s\S]*?\]);", raw_timings_js)
if not match:
    raise ValueError("Could not find exported timings array in timings.js")

array_text = match.group(1)
# Convert JS object keys to JSON keys, then parse
array_text = re.sub(r"(\{|,)\s*([A-Za-z_][A-Za-z0-9_]*)\s*:", r'\1 "\2":', array_text)
array_text = re.sub(r",\s*([\]}])", r"\1", array_text)
timings = json.loads(array_text)

state_map = {
    "left": "left_talking",
    "right": "right_talking",
    "quiet": "both_quiet",
}

timeline = []
for segment in timings:
    duration_seconds = segment["end"] - segment["start"]
    rounded_duration = round(duration_seconds, 1)
    frame_count = int(round(rounded_duration * FPS))
    timeline.extend([state_map[segment["state"]]] * frame_count)

# Clean old frames
for f in OUT_DIR.glob("frame_*.png"):
    f.unlink()

# Preselect blink frames
blink_frames = set()
for i in range(10, len(timeline), random.randint(30, 55)):
    blink_frames.add(i)

for i, state in enumerate(timeline):
    # Blink override: both mouths closed
    if i in blink_frames:
        img = assets["left_blink"] if random.random() < 0.5 else assets["right_blink"]

    elif state == "left_talking":
        # mouth flap: open every other frame
        img = assets["left_open"] if i % 2 == 0 else assets["both_quiet"]

    elif state == "right_talking":
        img = assets["right_open"] if i % 2 == 0 else assets["both_quiet"]

    else:
        img = assets["both_quiet"]

    img.save(OUT_DIR / f"frame_{i:05d}.png")

# Encode video + audio
cmd = [
    "ffmpeg",
    "-y",
    "-framerate", str(FPS),
    "-i", str(OUT_DIR / "frame_%05d.png"),
    "-i", "anim2_audio.mp3",
    "-c:v", "libx264",
    "-pix_fmt", "yuv420p",
    "-c:a", "aac",
    "-shortest",
    "dialogue_animation.mp4",
]

subprocess.run(cmd, check=True)

print("Done: dialogue_animation.mp4")


Done: dialogue_animation.mp4


In [17]:
import math
import json
from typing import Tuple


def ffprobe_duration(path: str) -> float:
    cmd = [
        "ffprobe",
        "-v",
        "error",
        "-show_entries",
        "format=duration",
        "-of",
        "default=noprint_wrappers=1:nokey=1",
        path,
    ]
    result = subprocess.run(cmd, check=True, capture_output=True, text=True)
    return float(result.stdout.strip())


def ffprobe_size(path: str) -> Tuple[int, int]:
    cmd = [
        "ffprobe",
        "-v",
        "error",
        "-select_streams",
        "v:0",
        "-show_entries",
        "stream=width,height",
        "-of",
        "json",
        path,
    ]
    result = subprocess.run(cmd, check=True, capture_output=True, text=True)
    data = json.loads(result.stdout)
    stream = data["streams"][0]
    return int(stream["width"]), int(stream["height"])


# 1) Build dramatic_horn twice in a row
subprocess.run(
    [
        "ffmpeg",
        "-y",
        "-i",
        "dramatic_horn.mp3",
        "-i",
        "dramatic_horn.mp3",
        "-filter_complex",
        "[0:a][1:a]concat=n=2:v=0:a=1[a]",
        "-map",
        "[a]",
        "-c:a",
        "aac",
        "dramatic_horn_twice.m4a",
    ],
    check=True,
)

# 2) Create a still segment from cold_weather.png that lasts exactly
#    for the duplicated horn audio (rounded to whole frames at FPS=12).
horn_duration = ffprobe_duration("dramatic_horn_twice.m4a")
extra_frames = int(math.ceil(horn_duration * FPS))
still_duration = extra_frames / FPS

width, height = ffprobe_size("dialogue_animation.mp4")

subprocess.run(
    [
        "ffmpeg",
        "-y",
        "-loop",
        "1",
        "-framerate",
        str(FPS),
        "-i",
        "cold_weather.png",
        "-i",
        "dramatic_horn_twice.m4a",
        "-vf",
        f"scale={width}:{height},format=yuv420p",
        "-t",
        f"{still_duration:.6f}",
        "-r",
        str(FPS),
        "-c:v",
        "libx264",
        "-pix_fmt",
        "yuv420p",
        "-c:a",
        "aac",
        "-shortest",
        "cold_tail_with_horn.mp4",
    ],
    check=True,
)

# 3) Concatenate original dialogue clip + cold tail with horn audio
subprocess.run(
    [
        "ffmpeg",
        "-y",
        "-i",
        "dialogue_animation.mp4",
        "-i",
        "cold_tail_with_horn.mp4",
        "-filter_complex",
        "[0:v:0][0:a:0][1:v:0][1:a:0]concat=n=2:v=1:a=1[v][a]",
        "-map",
        "[v]",
        "-map",
        "[a]",
        "-c:v",
        "libx264",
        "-pix_fmt",
        "yuv420p",
        "-c:a",
        "aac",
        "dialogue_plus_cold_weather.mp4",
    ],
    check=True,
)

print(f"Done: dialogue_plus_cold_weather.mp4 (added {extra_frames} cold-weather frames)")

Done: dialogue_plus_cold_weather.mp4 (added 62 cold-weather frames)


In [18]:
from pathlib import Path
import json
import re
import subprocess

BASE_VIDEO = Path("dialogue_plus_cold_weather.mp4")
SUBTITLES_JS = Path("subtitles_raw.js")
ASS_FILE = Path("subtitles_top.ass")
OUTPUT_VIDEO = Path("dialogue_plus_cold_weather_with_subtitles.mp4")


def ffprobe_video_meta(path: Path):
    cmd = [
        "ffprobe",
        "-v",
        "error",
        "-select_streams",
        "v:0",
        "-show_entries",
        "stream=width,height:format=duration",
        "-of",
        "json",
        str(path),
    ]
    result = subprocess.run(cmd, check=True, capture_output=True, text=True)
    data = json.loads(result.stdout)
    stream = data["streams"][0]
    width = int(stream["width"])
    height = int(stream["height"])
    duration = float(data["format"]["duration"])
    return width, height, duration


def seconds_to_ass_time(seconds: float) -> str:
    total_cs = int(round(seconds * 100))
    cs = total_cs % 100
    total_seconds = total_cs // 100
    s = total_seconds % 60
    total_minutes = total_seconds // 60
    m = total_minutes % 60
    h = total_minutes // 60
    return f"{h}:{m:02d}:{s:02d}.{cs:02d}"


def ass_escape(text: str) -> str:
    return (
        text.replace("\\", r"\\")
        .replace("{", r"\{")
        .replace("}", r"\}")
        .replace("\n", r"\N")
    )


# Parse exported subtitles array from subtitles_raw.js
raw_js = SUBTITLES_JS.read_text(encoding="utf-8")
match = re.search(r"export\s+const\s+subtitlesRaw\s*=\s*(\[[\s\S]*?\]);", raw_js)
if not match:
    raise ValueError("Could not find exported subtitlesRaw array in subtitles_raw.js")

array_text = match.group(1)
array_text = re.sub(r"(\{|,)\s*([A-Za-z_][A-Za-z0-9_]*)\s*:", r'\1 "\2":', array_text)
array_text = re.sub(r",\s*([\]}])", r"\1", array_text)
subtitles = json.loads(array_text)

width, height, duration = ffprobe_video_meta(BASE_VIDEO)

header = f"""[Script Info]
ScriptType: v4.00+
PlayResX: {width}
PlayResY: {height}
ScaledBorderAndShadow: yes

[V4+ Styles]
Format: Name, Fontname, Fontsize, PrimaryColour, SecondaryColour, OutlineColour, BackColour, Bold, Italic, Underline, StrikeOut, ScaleX, ScaleY, Spacing, Angle, BorderStyle, Outline, Shadow, Alignment, MarginL, MarginR, MarginV, Encoding
Style: TopText,Arial,56,&H00FFFFFF,&H00FFFFFF,&H00000000,&H64000000,0,0,0,0,100,100,0,0,1,2,0,8,60,60,40,1

[Events]
Format: Layer, Start, End, Style, Name, MarginL, MarginR, MarginV, Effect, Text
"""

events = []
for item in subtitles:
    start = float(item["start"])
    end = float(item["end"])
    if end <= 0 or start >= duration:
        continue
    start = max(0.0, start)
    end = min(duration, end)
    if end <= start:
        continue

    fi_text = ass_escape(item.get("fi", "").strip())
    en_text = ass_escape(item.get("en", "").strip())
    text = fi_text if not en_text else f"{fi_text}\\N{en_text}"

    events.append(
        f"Dialogue: 0,{seconds_to_ass_time(start)},{seconds_to_ass_time(end)},TopText,,0,0,0,,{text}"
    )

ASS_FILE.write_text(header + "\n".join(events) + "\n", encoding="utf-8")

# Burn subtitles directly into base video (no transparent intermediate overlay)
subprocess.run(
    [
        "ffmpeg",
        "-y",
        "-i",
        str(BASE_VIDEO),
        "-vf",
        f"subtitles={ASS_FILE.as_posix()}",
        "-c:v",
        "libx264",
        "-pix_fmt",
        "yuv420p",
        "-c:a",
        "copy",
        str(OUTPUT_VIDEO),
    ],
    check=True,
)

print(f"Done: {OUTPUT_VIDEO} (burned-in top subtitles)")

Done: dialogue_plus_cold_weather_with_subtitles.mp4 (burned-in top subtitles)


In [ ]:
from pathlib import Path

final_with_subs = Path("dialogue_plus_cold_weather_with_subtitles.mp4")

if final_with_subs.exists():
    print(f"Already created in previous cell: {final_with_subs}")
else:
    raise FileNotFoundError(
        "Run previous cell first to build dialogue_plus_cold_weather_with_subtitles.mp4"
    )

Already created in previous cell: dialogue_plus_cold_weather_with_subtitles.mp4


: 